# Discount analysis


In [3]:
from working_with_db import *

conn = connect_to_local_postgresql(db_name='postgres', user='jet', password='brains')

customer_status = ['new customer',
                   'existing customer: new product use',
                   'existing customer: renewal',
                   'existing customer: additional new licenses']

def calculate_value_for_different_customer(df, status):
    print(f"\nCalculating values for customer status - {status}")
    filtered_df = df[df['customer_status'] == status].drop(
        columns=['customer_status'])
    print(filtered_df)

Connection to DB successful


### General info
___

In [4]:
share_of_sales_df = execute_query_to_dataframe(conn, read_sql_file_to_string('discounts_sql/share_of_sales.sql'))
comparison_of_groups_df = execute_query_to_dataframe(conn,
                                                     read_sql_file_to_string('discounts_sql/comparison_of_groups.sql'))

print(f"\nThe share of sales with discounts is {share_of_sales_df['discount_usage_rate'][0]}% of all sales.")
print("It's a big part of sales, but usually it's sales without an advertising campaign(discount_id).")

sales_with_discount_id_df = execute_query_to_dataframe(conn, read_sql_file_to_string('discounts_sql/sales_with_discount_id.sql'))

print(f"\nThe share of sales with discount_id is {sales_with_discount_id_df['share_of_sales_with_discount_id'][0]}% of all sales.")

print(f"\nDiscounted products - {comparison_of_groups_df['discounted_product'][0]}. "
      f"All products - {comparison_of_groups_df['all_product'][0]}.")
print(f"Discounted license types - {comparison_of_groups_df['discounted_license_type'][0]}. "
      f"All license types - {comparison_of_groups_df['all_license_type'][0]}.")
print(f"Discounted customer status - {comparison_of_groups_df['discounted_customer_status'][0]}. "
      f"All customer status - {comparison_of_groups_df['all_customer_status'][0]}.")
print('The discounts apply to all user groups, products, and licenses.')


The share of sales with discounts is 64.13% of all sales.
It's a big part of sales, but usually it's sales without an advertising campaign(discount_id).

The share of sales with discount_id is 9.94% of all sales.

Discounted products - 13. All products - 13.
Discounted license types - 3. All license types - 3.
Discounted customer status - 4. All customer status - 4.
The discounts apply to all user groups, products, and licenses.


### Average sum amount
___
Let's divide all sales into two groups: with a discount and without a discount, and calculate the average sum amount.

In [5]:
print(execute_query_to_dataframe(conn, read_sql_file_to_string('discounts_sql/average_sum_amount.sql')))
print("\nℹ️ Discounts increase the average amount, but let's continue the research")

  avg_amount_discounted avg_amount_non_discounted
0                 68.61                     56.60

ℹ️ Discounts increase the average amount, but let's continue the research


In [6]:
average_sum_amount_for_different_customer_df = execute_query_to_dataframe(conn, read_sql_file_to_string(
    'discounts_sql/average_sum_amount_for_different_customer.sql'))

calculate_value_for_different_customer(average_sum_amount_for_different_customer_df, customer_status[0])
print('\nℹ️ For new customers, the discounts reduce the average amount.')


Calculating values for customer status - new customer
   sales_year discount_category avg_order_value
0        2018     With Discount           24.40
7        2018  Without Discount          135.84
11       2019     With Discount            7.46
14       2019  Without Discount          125.08
17       2020     With Discount           16.19
22       2020  Without Discount          119.66

ℹ️ For new customers, the discounts reduce the average amount.


In [7]:
calculate_value_for_different_customer(average_sum_amount_for_different_customer_df, customer_status[1])
print('\nℹ️ Discounts are effective for customers who use a new product.')


Calculating values for customer status - existing customer: new product use
   sales_year discount_category avg_order_value
1        2018     With Discount          119.09
5        2018  Without Discount          119.99
8        2019     With Discount          116.16
15       2019  Without Discount          104.95
18       2020     With Discount           51.83
21       2020  Without Discount           99.40

ℹ️ Discounts are effective for customers who use a new product.


In [8]:
calculate_value_for_different_customer(average_sum_amount_for_different_customer_df, customer_status[2])
print('\nℹ️ For renewal, discounts have a good effect.')


Calculating values for customer status - existing customer: renewal
   sales_year discount_category avg_order_value
2        2018     With Discount           73.40
4        2018  Without Discount           21.03
9        2019     With Discount           69.90
12       2019  Without Discount           20.32
16       2020     With Discount           68.29
20       2020  Without Discount           19.80

ℹ️ For renewal, discounts have a good effect.


In [9]:
calculate_value_for_different_customer(average_sum_amount_for_different_customer_df, customer_status[3])
print('\nℹ️ Interesting fact: Existing customer, additional new licenses, 2020.\n'
      'The large difference between the average sum with discount and without. Further research is needed.')


Calculating values for customer status - existing customer: additional new licenses
   sales_year discount_category avg_order_value
3        2018     With Discount           13.75
6        2018  Without Discount           86.77
10       2019     With Discount           28.14
13       2019  Without Discount           98.82
19       2020     With Discount           13.53
23       2020  Without Discount           67.25

ℹ️ Interesting fact: Existing customer, additional new licenses, 2020.
The large difference between the average sum with discount and without. Further research is needed.


### Average discount size
___
Let's take only sales with discounts and without promotions and group them by products.

In [10]:
average_discount_size_df = execute_query_to_dataframe(conn, read_sql_file_to_string('discounts_sql/average_discount_size.sql'))
calculate_value_for_different_customer(average_discount_size_df, customer_status[0])
print("\nℹ️ Product I is free for new customers or has a trial period.")


Calculating values for customer status - new customer
   product_code avg_discount avg_amount total_discount total_amount
0             X       224.51      39.29        4939.24       864.29
6             L        20.85     118.15          20.85       118.15
9             K        22.18      88.70          22.18        88.70
11            I       174.36       0.00         174.36         0.00
14            H       139.53      32.93         558.12       131.72
20            G        13.35      75.65          13.35        75.65
23            F        14.79      83.80          14.79        83.80
28            D         9.73      16.31          29.19        48.94
29            C        81.33      17.25         325.31        69.00
33            B        61.55      36.27         123.10        72.54
38            A        33.03     132.12          66.06       264.24

ℹ️ Product I is free for new customers or has a trial period.


In [11]:
calculate_value_for_different_customer(average_discount_size_df, customer_status[1])


Calculating values for customer status - existing customer: new product use
   product_code avg_discount avg_amount total_discount total_amount
1             X        96.59     106.02       80458.86     88317.02
4             L        81.24      15.15         893.61       166.69
7             K        36.73      62.73         183.63       313.63
12            I       152.78       8.22       53627.34      2884.43
16            H        52.64      26.91        1000.23       511.29
18            G        85.53      21.37          85.53        21.37
21            F        20.81      57.91          83.22       231.65
25            E        20.95      78.38          20.95        78.38
26            D        41.84      18.33         334.71       146.60
30            C        62.00      23.83         434.03       166.82
34            B        44.25      32.59         486.71       358.47
36            A        67.07      73.28        3554.68      3884.01


In [12]:
calculate_value_for_different_customer(average_discount_size_df, customer_status[2])


Calculating values for customer status - existing customer: renewal
   product_code avg_discount avg_amount total_discount total_amount
2             X        71.82     113.97      439055.32    696712.81
5             L        33.89      54.14        6235.08      9961.88
8             K        29.34      41.35        3667.70      5168.30
10            J        41.44      55.51         787.35      1054.60
13            I        41.22      71.32       12077.10     20895.71
17            H        40.52      65.36       56155.91     90584.63
19            G        29.66      48.94        5932.10      9788.52
22            F        17.28      30.64        9212.06     16333.54
24            E        21.05      32.24        4968.05      7609.54
27            D        15.51      25.38       25849.68     42312.46
31            C        25.63      40.07       41911.27     65514.59
35            B        27.23      44.80      101461.86    166939.24
37            A        45.00      74.67      23

In [13]:
calculate_value_for_different_customer(average_discount_size_df, customer_status[3])
print("\nℹ️ Additional new licenses in this products is free.")


Calculating values for customer status - existing customer: additional new licenses
   product_code avg_discount avg_amount total_discount total_amount
3             X        26.96       0.00         377.40         0.00
15            H        98.94       0.00         197.87         0.00
32            B        98.96       0.00          98.96         0.00
39            A        16.90       0.00          16.90         0.00

ℹ️ Additional new licenses in this products is free.


In [14]:
discount_size_and_payment_amounts_anomalies_df = execute_query_to_dataframe(conn, read_sql_file_to_string('discounts_sql/discount_size_and_payment_amounts_anomalies.sql'))

print("\nLet's get only new customers with discount (but without discount_id) who bought product X:")
print(discount_size_and_payment_amounts_anomalies_df)
print("\n❓Why 4 customers payed to this product?")


Let's get only new customers with discount (but without discount_id) who bought product X:
   customer processed_date  discount_in_usd  amount_in_usd
0   3343099     2018-05-22           293.58           0.00
1   3345743     2018-05-23           293.58           0.00
2   3346659     2018-05-23           293.58           0.00
3   3413866     2018-06-05           290.57           0.00
4   3484727     2018-07-03           291.42           0.00
5   3566500     2018-08-01           291.54           0.00
6   3596556     2018-08-11            73.17         212.08
7   3947468     2018-11-12           282.52           0.00
8   4216729     2019-02-05           284.98           0.00
9   4223636     2019-02-07            28.37           0.00
10  4440642     2019-04-09            28.00           0.00
11  4617952     2019-06-05           279.98           0.00
12  4924051     2019-09-16           276.29           0.00
13  5608423     2020-02-18           269.79           0.00
14  5797052     2020-03

In [15]:
anomalies_negative_discount_df = execute_query_to_dataframe(conn, read_sql_file_to_string('discounts_sql/anomalies_negative_discout.sql'))
print(anomalies_negative_discount_df)
print("\n❓Why this sales have negative discount size?")

    customer processed_date  amount_in_usd  discount_in_usd
0     317707     2019-06-13         675.98          -394.04
1    6325703     2020-08-07         338.78          -162.15
2     257746     2019-05-02         255.41           -88.26
3     366798     2020-09-28         169.40           -65.86
4     197879     2018-06-12         235.50           -59.96
..       ...            ...            ...              ...
99   1984912     2019-11-25         164.93            -0.17
100    72162     2020-11-16          69.87            -0.17
101  1475221     2018-09-03         103.87            -0.17
102  1686344     2018-09-25          10.58            -0.11
103   351033     2018-05-14          10.70            -0.04

[104 rows x 4 columns]

❓Why this sales have negative discount size?


### Promotions
___
Let's look at sales from promotions.


In [16]:
number_of_promotions_df = execute_query_to_dataframe(conn, read_sql_file_to_string('discounts_sql/number_of_products_in_promotions.sql'))

print("\nNumber of products in promotions:")
print("Only 1/10 of all promotions contains more then 1 product.\n")
print(number_of_promotions_df)


Number of products in promotions:
Only 1/10 of all promotions contains more then 1 product.

    discount_id  number_of_products avg_discount_percentage
0          1162                  13                      28
1          3868                  11                      52
2           795                  10                     100
3          3132                   8                     100
4          3131                   6                     100
..          ...                 ...                     ...
110        2059                   1                     100
111        2119                   1                     100
112        2192                   1                      46
113        2362                   1                      20
114        2476                   1                     100

[115 rows x 3 columns]


In [64]:
number_of_promotions_with_product_df = execute_query_to_dataframe(conn, read_sql_file_to_string('discounts_sql/number_of_promotions_with_product.sql'))

print("\nMost often, products", number_of_promotions_with_product_df['product_code'][0], ",", number_of_promotions_with_product_df['product_code'][1], ", and ", number_of_promotions_with_product_df['product_code'][2], " participate in promotions.")
print(number_of_promotions_with_product_df)


Most often, products C , X , and  A  participate in promotions.
   product_code  number_of_promotions_with_product
0             C                                 38
1             X                                 28
2             A                                 26
3             B                                 17
4             F                                 14
5             L                                 11
6             D                                  9
7             H                                  6
8             K                                  6
9             I                                  4
10            G                                  4
11            E                                  4
12            J                                  1
